# QMOOD Tabanli Yazilim Kalitesi Analizi — jsoup

**Yazilim Mimarisi ve Tasarim Yontemleri — Donem Projesi**

Bu defter, secilen acik kaynak yazilimin (**jsoup**) farkli surumlerinden CK araci
(`mauricioaniche/ck`) ile cikarilan `class.csv` metriklerini kullanarak:

1. Her surum icin **11 QMOOD tasarim ozelligini** hesaplar,
2. Ozellikleri baz surume gore **normalize** eder,
3. Dersteki **QMOOD formulleriyle** 6 kalite niteligini + Toplam Kalite Indeksi (TQI) hesaplar,
4. Surumler arasi kalite degisimini **gorsellestirir**,
5. **Mimari bozulma** ve **teknik borc** belirtilerini veriden otomatik isaretler.

> Rapor bolumleriyle eslesme: *Yontem* → 1-2, *Analiz Sureci* → 3-4, *Sonuclar* → 5-6, *Tartisma* → son hucredeki yorum notlari.


## 1. Yontem — Metrik Eslemesi

QMOOD modeli (Bansiya & Davis, 2002), dusuk duzey tasarim metriklerini ust duzey
kalite niteliklerine baglar. Tasarim ozelligi → CK metrigi eslemesi:

| QMOOD tasarim ozelligi | Kullanilan deger | Kaynak |
|---|---|---|
| Design Size (DSC) | sinif sayisi | CK — dogrudan |
| Complexity (NOM) | ort. `totalMethodsQty` | CK — dogrudan |
| Messaging (CIS) | ort. `publicMethodsQty` | CK — dogrudan |
| Coupling (DCC) | ort. `cbo` | CK — dogrudan |
| Abstraction (ANA) | ort. `dit` | CK — dogrudan |
| Encapsulation (DAM) | ort. (private+protected)/toplam alan | CK — turetilen |
| Hierarchies (NOH) | kalitim koku sayisi | CK — turetilen (yaklasik) |
| **Cohesion (CAM)** | ort. `1 - lcom*` | **yaklasik** |
| **Polymorphism (NOP)** | ort. override edilebilir metot | **yaklasik** |
| **Inheritance (MFA)** | `dit` tabanli kalitilan islevsellik orani | **yaklasik** |
| **Composition (MOA)** | ort. ornek (instance) alan sayisi | **yaklasik (kaba)** |

**Onemli (raporda aciklanmali):** Son 4 ozellik CK ciktisinda dogrudan bulunmaz; makul
proxy'lerle hesaplanmistir. Ozellikle **MFA** ve **MOA** en zayif noktalardir (alan tipi /
kalitilan metot bilgisi CK'da yoktur). Daha yuksek dogruluk istenirse bu iki ozellik
Designite degerleriyle degistirilebilir — `design_properties` fonksiyonundaki ilgili
satirlari guncellemek yeterlidir.

### QMOOD kalite niteligi formulleri (Buzluca, *Yazilim Tasarimi Kalitesi*, slayt 5.14)

- **Reusability** = +0.5·DesignSize −0.25·Coupling +0.25·Cohesion +0.5·Messaging
- **Flexibility** = +0.25·Encapsulation −0.25·Coupling +0.5·Composition +0.5·Polymorphism
- **Understandability** = −0.33·(DesignSize + Abstraction + Coupling + Polymorphism + Complexity) +0.33·(Encapsulation + Cohesion)
- **Functionality** = +0.22·DesignSize +0.22·Hierarchies +0.12·Cohesion +0.22·Polymorphism +0.22·Messaging
- **Extendibility** = +0.5·Abstraction −0.5·Coupling +0.5·Inheritance +0.5·Polymorphism
- **Effectiveness** = +0.2·(Abstraction + Encapsulation + Composition + Inheritance + Polymorphism)
- **TQI** = altı niteligin toplami


### Tasinabilir kurulum (GitHub icin)

Bu defter sabit bir bilgisayar yoluna **bagli degildir**; veri klasorunu otomatik bulur.
Repoyu klonlayan herkeste calissin diye onerilen yapi:

```
<repo>/
├── qmood_analiz.ipynb
└── Sonuclar/
    ├── 1.13.1/class.csv
    ├── 1.14.1/class.csv
    └── ...
```

`Sonuclar` klasorunu (icindeki `class.csv`'lerle) repoya ekle. Defter, calistirildigi
dizinden yukari dogru `Sonuclar` klasorunu arar. Farkli bir konum gerekirse
`QMOOD_BASE_DIR` ortam degiskeniyle elle belirtebilirsin.

In [ ]:
import os, re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ===================== AYARLAR (TASINABILIR — yol otomatik bulunur) =====================
DATA_FOLDER_NAME = "Sonuclar"   # repodaki veri klasorunun adi

BASE_DIR = os.environ.get("QMOOD_BASE_DIR")          # 1) istege bagli elle override
if not BASE_DIR:
    start = Path.cwd()
    for cand in [start, *start.parents]:             # 2) yukari dogru 'Sonuclar' ara
        if (cand / DATA_FOLDER_NAME).is_dir():
            BASE_DIR = str(cand / DATA_FOLDER_NAME)
            break
    else:
        BASE_DIR = DATA_FOLDER_NAME                   # 3) son care: goreli yol
BASE_DIR = str(Path(BASE_DIR))

OUTPUT_DIR = os.path.join(BASE_DIR, "_qmood_sonuc")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Veri klasoru :", BASE_DIR)
print("Cikti klasoru:", OUTPUT_DIR)

In [ ]:
def version_key(v):
    """ '1.14.1' -> [1,14,1]  (dogru sayisal siralama icin) """
    return [int(x) for x in re.findall(r"\d+", v)]


def design_properties(df):
    """ Bir surumun class.csv'sinden proje duzeyi 11 QMOOD tasarim ozelligi. """
    n = len(df)
    mean = lambda c: float(df[c].mean()) if n else 0.0

    # --- DOGRUDAN / TURETILEN ---
    DSC = float(n)                  # Design Size
    NOM = mean("totalMethodsQty")   # Complexity
    CIS = mean("publicMethodsQty")  # Messaging
    DCC = mean("cbo")               # Coupling
    ANA = mean("dit")               # Abstraction

    fld = df[df["totalFieldsQty"] > 0]                       # Encapsulation
    DAM = float(((fld["privateFieldsQty"] + fld["protectedFieldsQty"])
                 / fld["totalFieldsQty"]).mean()) if len(fld) else 0.0

    NOH = float(((df["noc"] > 0) & (df["dit"] <= 1)).sum())  # Hierarchies (yaklasik)

    # --- YAKLASIK (CK'da dogrudan yok) ---
    lcomn = df["lcom*"][(df["lcom*"] >= 0) & (df["lcom*"] <= 1)]
    CAM = float((1 - lcomn).mean()) if len(lcomn) else 0.0   # Cohesion ~ 1 - lcom*

    NOP = float((df["publicMethodsQty"] + df["protectedMethodsQty"]
                 - df["finalMethodsQty"] - df["staticMethodsQty"]).clip(lower=0).mean())  # Polymorphism

    dit_safe = df["dit"].replace(0, 1)
    MFA = float(np.where(df["dit"] >= 1, (df["dit"] - 1) / dit_safe, 0).mean()) if n else 0.0  # Inheritance

    MOA = float((df["totalFieldsQty"] - df["staticFieldsQty"]).clip(lower=0).mean())  # Composition (kaba)

    return {"DSC": DSC, "NOH": NOH, "ANA": ANA, "DAM": DAM, "DCC": DCC,
            "CAM": CAM, "MOA": MOA, "MFA": MFA, "NOP": NOP, "CIS": CIS, "NOM": NOM}


def quality_attributes(p):
    """ QMOOD kalite niteligi formulleri (Buzluca 5.14). Girdi: normalize tasarim ozellikleri. """
    q = {}
    q["Reusability"]       =  0.50*p["DSC"] - 0.25*p["DCC"] + 0.25*p["CAM"] + 0.50*p["CIS"]
    q["Flexibility"]       =  0.25*p["DAM"] - 0.25*p["DCC"] + 0.50*p["MOA"] + 0.50*p["NOP"]
    q["Understandability"] = -0.33*p["DSC"] - 0.33*p["ANA"] + 0.33*p["DAM"] - 0.33*p["DCC"] \
                             + 0.33*p["CAM"] - 0.33*p["NOP"] - 0.33*p["NOM"]
    q["Functionality"]     =  0.22*p["DSC"] + 0.22*p["NOH"] + 0.12*p["CAM"] + 0.22*p["NOP"] + 0.22*p["CIS"]
    q["Extendibility"]     =  0.50*p["ANA"] - 0.50*p["DCC"] + 0.50*p["MFA"] + 0.50*p["NOP"]
    q["Effectiveness"]     =  0.20*p["ANA"] + 0.20*p["DAM"] + 0.20*p["MOA"] + 0.20*p["MFA"] + 0.20*p["NOP"]
    q["TQI"] = sum(q.values())
    return q

## 2. Analiz Sureci — Verilerin Yuklenmesi ve Tasarim Ozellikleri

In [ ]:
versions = [d for d in os.listdir(BASE_DIR)
            if os.path.isfile(os.path.join(BASE_DIR, d, "class.csv"))]
versions.sort(key=version_key)
assert versions, "class.csv bulunamadi! BASE_DIR ve klasor yapisini kontrol edin."
print(f"{len(versions)} surum bulundu:", versions)

# --- Istege bagli: MFA ve MOA en zayif iki yaklasik metriktir. Designite ile olcersen
#     daha dogru degerleri buraya yaz; bos birakilirsa CK tabanli yaklasik kullanilir. ---
DESIGNITE_OVERRIDE = {
    # "1.13.1": {"MFA": 0.41, "MOA": 1.20},
    # "1.14.1": {"MFA": 0.43, "MOA": 1.31},
}

raw = pd.DataFrame(
    [{"version": v, **design_properties(pd.read_csv(os.path.join(BASE_DIR, v, "class.csv")))}
     for v in versions]
).set_index("version")

for _v, _ov in DESIGNITE_OVERRIDE.items():          # override uygula (varsa)
    if _v in raw.index:
        for _k, _val in _ov.items():
            raw.loc[_v, _k] = _val

raw.to_csv(os.path.join(OUTPUT_DIR, "tasarim_ozellikleri_ham.csv"))
raw.round(3)

### Normalizasyon
Tasarim ozelliklerinin olcekleri cok farklidir (DSC yuzlerce, DAM 0-1). Karsilastirilabilir
olmasi icin her ozellik **ilk (baz) surume** bolunur; boylece ilk surum = 1.0 ve sonraki
surumler bagil degisimi gosterir.

In [ ]:
base = raw.iloc[0].replace(0, np.nan)
norm = raw.divide(base).fillna(0.0)
norm.to_csv(os.path.join(OUTPUT_DIR, "tasarim_ozellikleri_normalize.csv"))
norm.round(3)

### QMOOD Kalite Nitelikleri

In [ ]:
qdf = pd.DataFrame(
    [{"version": v, **quality_attributes(norm.loc[v].to_dict())} for v in versions]
).set_index("version")
qdf.to_csv(os.path.join(OUTPUT_DIR, "qmood_kalite_nitelikleri.csv"))
qdf.round(3)

### Surum Kalite Siralamasi (TQI)
Surumler toplam kalite indeksine gore siralanir (yuksek TQI = daha iyi tasarim kalitesi).

In [ ]:
ranking = qdf[["TQI"]].copy()
ranking["sira"] = ranking["TQI"].rank(ascending=False).astype(int)
ranking = ranking.sort_values("TQI", ascending=False)
ranking.to_csv(os.path.join(OUTPUT_DIR, "tqi_siralama.csv"))
ranking.round(3)

## 3. Gorsellestirme

### 3.1 Kalite niteliklerinin surumlere gore degisimi

In [ ]:
ATTRS = ["Reusability", "Flexibility", "Understandability",
         "Functionality", "Extendibility", "Effectiveness"]
plt.figure(figsize=(11, 6))
for col in ATTRS:
    plt.plot(qdf.index, qdf[col], marker="o", linewidth=2, label=col)
plt.title("QMOOD Kalite Niteliklerinin Surumlere Gore Degisimi")
plt.xlabel("Surum"); plt.ylabel("Kalite degeri (baz surume gore)")
plt.xticks(rotation=45); plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_kalite_nitelikleri.png"), dpi=150)
plt.show()

### 3.2 Toplam Kalite Indeksi (TQI)

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(qdf.index, qdf["TQI"], marker="s", linewidth=2, color="#333")
plt.fill_between(range(len(qdf)), qdf["TQI"], alpha=0.1, color="#333")
plt.title("Toplam Kalite Indeksi (TQI) Degisimi")
plt.xlabel("Surum"); plt.ylabel("TQI"); plt.xticks(rotation=45)
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_tqi.png"), dpi=150)
plt.show()

### 3.3 Tasarim ozellikleri isi haritasi (normalize)
Her ozelligin surumler boyunca bagil degisimini tek bakista gosterir.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
data = norm.T  # satir=ozellik, sutun=surum
im = ax.imshow(data.values, aspect="auto", cmap="RdYlGn")
ax.set_xticks(range(len(data.columns))); ax.set_xticklabels(data.columns, rotation=45)
ax.set_yticks(range(len(data.index)));   ax.set_yticklabels(data.index)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j, i, f"{data.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, label="Baz surume gore oran")
plt.title("Tasarim Ozellikleri Isi Haritasi (normalize)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_isi_haritasi.png"), dpi=150)
plt.show()

### 3.4 Ilk ve son surumun radar karsilastirmasi

In [ ]:
labels = ATTRS
first_v, last_v = versions[0], versions[-1]
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for v, color in [(first_v, "#1f77b4"), (last_v, "#d62728")]:
    vals = qdf.loc[v, labels].tolist(); vals += vals[:1]
    ax.plot(angles, vals, linewidth=2, label=v, color=color)
    ax.fill(angles, vals, alpha=0.1, color=color)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels)
ax.set_title(f"Kalite Profili: {first_v} vs {last_v}")
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_radar.png"), dpi=150)
plt.show()

### 3.5 Ilk → son surum kalite degisimi (%)

In [ ]:
pct = ((qdf.loc[last_v, ATTRS] - qdf.loc[first_v, ATTRS]) /
       qdf.loc[first_v, ATTRS].abs().replace(0, np.nan) * 100)
plt.figure(figsize=(10, 5))
colors = ["#2ca02c" if x >= 0 else "#d62728" for x in pct]
plt.bar(ATTRS, pct.values, color=colors)
plt.axhline(0, color="black", linewidth=0.8)
plt.title(f"Kalite Niteliklerinde Degisim: {first_v} -> {last_v} (%)")
plt.ylabel("% degisim"); plt.xticks(rotation=30); plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_yuzde_degisim.png"), dpi=150)
plt.show()
pct.round(1)

### 3.6 Teknik borc ile ilgili ham metriklerin trendi
Buyume (DSC), baglasim (DCC), karmasiklik (NOM) ve kohezyon (CAM) — mimari bozulma yorumu icin.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col, title in zip(axes.ravel(),
                          ["DSC", "DCC", "NOM", "CAM"],
                          ["Tasarim Boyutu (sinif sayisi)", "Baglasim (ort. CBO)",
                           "Karmasiklik (ort. metot)", "Kohezyon (1 - lcom*)"]):
    ax.plot(raw.index, raw[col], marker="o", linewidth=2)
    ax.set_title(title); ax.set_xlabel("Surum"); ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("Teknik Borc / Mimari Bozulma Gostergeleri (ham degerler)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_teknik_borc_trend.png"), dpi=150)
plt.show()

### 3.7 Ardisik (bir onceki) surumle kiyaslama
Her surumun **bir onceki** surume gore kalite degisimi (mutlak fark = guncel − onceki).
Pozitif/yesil = iyilesme, negatif/kirmizi = bozulma. Hangi surum gecislerinin kaliteyi
artirdigini/dusurdugunu gosterir (ilk↔son kiyasi 3.4-3.5'te ayrica duruyor).

In [ ]:
# Bir onceki surume gore MUTLAK degisim (QMOOD degerleri negatif olabildigi icin % yerine fark)
consec = qdf[ATTRS + ["TQI"]].diff().iloc[1:].round(3)
consec.index = [f"{versions[i-1]} -> {versions[i]}" for i in range(1, len(versions))]
consec.to_csv(os.path.join(OUTPUT_DIR, "ardisik_degisim.csv"))
consec

In [ ]:
# Ardisik degisim isi haritasi (gecisler x nitelikler)
m = consec[ATTRS].T
vmax = float(np.nanmax(np.abs(m.values))) or 1.0
fig, ax = plt.subplots(figsize=(max(8, 1.1*len(consec)), 5))
im = ax.imshow(m.values, aspect="auto", cmap="RdYlGn", vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(m.columns))); ax.set_xticklabels(m.columns, rotation=45, ha="right")
ax.set_yticks(range(len(m.index)));   ax.set_yticklabels(m.index)
for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        ax.text(j, i, f"{m.values[i, j]:+.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, label="Degisim (guncel - onceki)")
plt.title("Ardisik Surumler Arasi Kalite Degisimi")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "07_ardisik_degisim.png"), dpi=150)
plt.show()

In [ ]:
# TQI'nin ardisik degisimi (hangi gecis kaliteyi artirdi/dusurdu)
vals = consec["TQI"].values
colors = ["#2ca02c" if x >= 0 else "#d62728" for x in vals]
plt.figure(figsize=(max(8, 1.1*len(consec)), 5))
plt.bar(consec.index, vals, color=colors)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("TQI'nin Ardisik Surumler Arasi Degisimi")
plt.ylabel("TQI degisimi (guncel - onceki)")
plt.xticks(rotation=45, ha="right"); plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "08_tqi_ardisik.png"), dpi=150)
plt.show()

## 4. Mimari Bozulma ve Teknik Borc Belirtileri (otomatik)
Asagidaki hucre veriden otomatik **isaretler** uretir. Bunlar yorum degil, yorumun
dayanagidir — raporun *Analiz Sureci / Tartisma* bolumunde bu isaretleri kendi
sozlerinle yorumlamalisin.

In [ ]:
def trend(series):
    chg = (series.iloc[-1] - series.iloc[0]) / (abs(series.iloc[0]) or 1) * 100
    return chg

print("=== Ilk surum -> Son surum degisimleri ===\n")
print(f"Tasarim boyutu (DSC) : %{trend(raw['DSC']):+.1f}")
print(f"Baglasim     (DCC)   : %{trend(raw['DCC']):+.1f}   (artis = teknik borc belirtisi)")
print(f"Karmasiklik  (NOM)   : %{trend(raw['NOM']):+.1f}   (artis = teknik borc belirtisi)")
print(f"Kohezyon     (CAM)   : %{trend(raw['CAM']):+.1f}   (dusus = teknik borc belirtisi)")
print()

flags = []
if trend(raw["DCC"]) > 0:  flags.append("Baglasim (DCC) artmis -> moduller arasi bagimlilik buyuyor.")
if trend(raw["NOM"]) > 0:  flags.append("Sinif basina metot (NOM) artmis -> siniflar sismis olabilir.")
if trend(raw["CAM"]) < 0:  flags.append("Kohezyon (CAM) azalmis -> sinif sorumluluklari dagilmis olabilir.")
if trend(raw["DCC"]) > 0 and trend(raw["CAM"]) < 0:
    flags.append("Baglasim artarken kohezyon dusmus -> klasik mimari bozulma kalibi.")
if qdf["Understandability"].iloc[-1] < qdf["Understandability"].iloc[0]:
    flags.append("Anlasilirlik (Understandability) dusmus -> bakim maliyeti artma egiliminde.")

print("=== Otomatik teknik-borc isaretleri ===")
if flags:
    for f in flags: print(" -", f)
else:
    print(" - Belirgin bozulma isareti bulunamadi.")

## 5. Sonuc ve Yorum (rapora yazilacak)

Bu hucreyi kendi yorumunla doldur:

- Hangi kalite nitelikleri zaman icinde **iyilesti / kotulesti**? (3.1, 3.5)
- TQI trendi ne anlatiyor? (3.2)
- Isi haritasinda hangi tasarim ozelligi en cok degisti? (3.3)
- Teknik-borc isaretleri (4) hangi surumlerde belirginlesti? Bir refactoring/buyuk surum ile ortusyor mu?
- Yaklasik hesaplanan 4 metrigin (CAM, NOP, MFA, MOA) sonuclara olasi etkisi nedir? (Yontem)

> Sonraki asama (bu projede simdilik ertelendi): metrik tablolarini en az 3 LLM'e verip
> kalite/bakim/teknik-borc/refactoring yorumlarini almak ve QMOOD sonuclarinla kiyaslamak.
